In [2]:
# ============================================================
# FORESIGHT - DATA PIPELINE
# STEP 1: PROJECT PATHS
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# PROJECT PATH
# ------------------------------------------------------------

# Your notebook is inside the notebook folder.
# parent = Foresight project folder

BASE_DIR = Path.cwd().parent

# ------------------------------------------------------------
# DATA FOLDERS
# ------------------------------------------------------------

RAW_DATA_DIR = BASE_DIR / "data" / "Raw"
PROCESSED_DATA_DIR = BASE_DIR / "data" / "Processed"

# Create Processed folder if it does not exist
PROCESSED_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# CHECK PATHS
# ------------------------------------------------------------

print("Project folder:")
print(BASE_DIR)

print("\nRaw data folder:")
print(RAW_DATA_DIR)

print("\nProcessed data folder:")
print(PROCESSED_DATA_DIR)

print("\nChecking raw files...")

print(
    "calendar.csv:",
    (RAW_DATA_DIR / "calendar.csv").exists()
)

print(
    "sales_train_validation.csv:",
    (RAW_DATA_DIR / "sales_train_validation.csv").exists()
)

print(
    "sell_prices.csv:",
    (RAW_DATA_DIR / "sell_prices.csv").exists()
)

print("\n✅ PATH SETUP COMPLETED")

Project folder:
c:\Users\SAHITYA\OneDrive\Desktop\Foresight 1

Raw data folder:
c:\Users\SAHITYA\OneDrive\Desktop\Foresight 1\data\Raw

Processed data folder:
c:\Users\SAHITYA\OneDrive\Desktop\Foresight 1\data\Processed

Checking raw files...
calendar.csv: True
sales_train_validation.csv: True
sell_prices.csv: True

✅ PATH SETUP COMPLETED


In [3]:
# ============================================================
# STEP 2: LOAD M5 RAW DATASETS
# ============================================================

print("=" * 60)
print("LOADING M5 DATASETS")
print("=" * 60)

# ------------------------------------------------------------
# LOAD CALENDAR
# ------------------------------------------------------------

calendar = pd.read_csv(
    RAW_DATA_DIR / "calendar.csv"
)

# ------------------------------------------------------------
# LOAD SALES
# ------------------------------------------------------------

sales = pd.read_csv(
    RAW_DATA_DIR / "sales_train_validation.csv"
)

# ------------------------------------------------------------
# LOAD SELL PRICES
# ------------------------------------------------------------

prices = pd.read_csv(
    RAW_DATA_DIR / "sell_prices.csv"
)

# ------------------------------------------------------------
# DISPLAY DATASET INFORMATION
# ------------------------------------------------------------

print("\nCalendar Dataset:")
print("Rows    :", calendar.shape[0])
print("Columns :", calendar.shape[1])

print("\nSales Dataset:")
print("Rows    :", sales.shape[0])
print("Columns :", sales.shape[1])

print("\nSell Prices Dataset:")
print("Rows    :", prices.shape[0])
print("Columns :", prices.shape[1])

print("\n✅ ALL M5 DATASETS LOADED SUCCESSFULLY")

LOADING M5 DATASETS

Calendar Dataset:
Rows    : 1969
Columns : 13

Sales Dataset:
Rows    : 30490
Columns : 1918

Sell Prices Dataset:
Rows    : 6841121
Columns : 4

✅ ALL M5 DATASETS LOADED SUCCESSFULLY


In [5]:
# ============================================================
# DATA QUALITY CHECK
# ============================================================

print("=" * 60)
print("DATA QUALITY CHECK")
print("=" * 60)

# Calendar
print("\nCalendar:")
print("Shape:", calendar.shape)
print("Missing values:", calendar.isnull().sum().sum())

# Sales
print("\nSales:")
print("Shape:", sales.shape)
print("Missing values:", sales.isnull().sum().sum())

# Sell Prices
print("\nSell Prices:")
print("Shape:", prices.shape)
print("Missing values:")
print(prices.isnull().sum())

print("\n" + "=" * 60)
print("DATA QUALITY CHECK COMPLETED")
print("=" * 60)

DATA QUALITY CHECK

Calendar:
Shape: (1969, 13)
Missing values: 7542

Sales:
Shape: (30490, 1918)
Missing values: 0

Sell Prices:
Shape: (6841121, 4)
Missing values:
store_id      0
item_id       0
wm_yr_wk      0
sell_price    0
dtype: int64

DATA QUALITY CHECK COMPLETED


In [6]:
# ============================================================
# STEP 3 — CLEAN CALENDAR DATA
# ============================================================

print("=" * 60)
print("CLEANING CALENDAR DATA")
print("=" * 60)

# Make a copy so the original raw data is not modified
calendar_clean = calendar.copy()

# ------------------------------------------------------------
# 1. Convert date column to datetime
# ------------------------------------------------------------

calendar_clean["date"] = pd.to_datetime(calendar_clean["date"])

# ------------------------------------------------------------
# 2. Fill missing event information
# ------------------------------------------------------------

event_columns = [
    "event_name_1",
    "event_type_1",
    "event_name_2",
    "event_type_2"
]

for col in event_columns:
    if col in calendar_clean.columns:
        calendar_clean[col] = calendar_clean[col].fillna("No Event")

# ------------------------------------------------------------
# 3. Check missing values after cleaning
# ------------------------------------------------------------

print("\nMissing values after cleaning:")

print(calendar_clean.isnull().sum())

# ------------------------------------------------------------
# 4. Check date range
# ------------------------------------------------------------

print("\nDate Range:")
print("Start Date:", calendar_clean["date"].min())
print("End Date  :", calendar_clean["date"].max())

# ------------------------------------------------------------
# 5. Check shape
# ------------------------------------------------------------

print("\nClean Calendar Shape:")
print(calendar_clean.shape)

print("\n" + "=" * 60)
print("CALENDAR CLEANING COMPLETED")
print("=" * 60)

CLEANING CALENDAR DATA

Missing values after cleaning:
date            0
wm_yr_wk        0
weekday         0
wday            0
month           0
year            0
event_name_1    0
event_type_1    0
event_name_2    0
event_type_2    0
snap_CA         0
snap_TX         0
snap_WI         0
dtype: int64

Date Range:
Start Date: 2011-01-29 00:00:00
End Date  : 2016-06-19 00:00:00

Clean Calendar Shape:
(1969, 13)

CALENDAR CLEANING COMPLETED


In [8]:
# ============================================================
# STEP 4 — CREATE DAILY AGGREGATED SALES
# ============================================================

print("=" * 60)
print("CREATING DAILY AGGREGATED SALES")
print("=" * 60)

# Identify daily sales columns
sales_cols = [
    col for col in sales.columns
    if col.startswith("d_")
]

print("\nNumber of daily columns:", len(sales_cols))

# ------------------------------------------------------------
# Sum sales across all items, stores and categories
# for each day
# ------------------------------------------------------------

daily_sales = sales[sales_cols].sum(axis=0)

# Convert Series to DataFrame
daily_sales_df = daily_sales.reset_index()

# Rename columns
daily_sales_df.columns = ["d", "daily_sales"]

print("\nDaily Sales Shape:")
print(daily_sales_df.shape)

print("\nFirst 10 rows:")
print(daily_sales_df.head(10))

print("\nLast 10 rows:")
print(daily_sales_df.tail(10))

print("\nMissing values:")
print(daily_sales_df["daily_sales"].isnull().sum())

print("\n" + "=" * 60)
print("DAILY SALES CREATED SUCCESSFULLY")
print("=" * 60)

CREATING DAILY AGGREGATED SALES

Number of daily columns: 1913

Daily Sales Shape:
(1913, 2)

First 10 rows:
      d  daily_sales
0   d_1        32631
1   d_2        31749
2   d_3        23783
3   d_4        25412
4   d_5        19146
5   d_6        29211
6   d_7        28010
7   d_8        37932
8   d_9        32736
9  d_10        25572

Last 10 rows:
           d  daily_sales
1903  d_1904        41789
1904  d_1905        48362
1905  d_1906        51640
1906  d_1907        38059
1907  d_1908        37570
1908  d_1909        35343
1909  d_1910        35033
1910  d_1911        40517
1911  d_1912        48962
1912  d_1913        49795

Missing values:
0

DAILY SALES CREATED SUCCESSFULLY


In [10]:
# ============================================================
# STEP 7 — MERGE DAILY SALES WITH CALENDAR
# ============================================================

print("=" * 60)
print("MERGING DAILY SALES WITH CALENDAR")
print("=" * 60)

# ------------------------------------------------------------
# 1. Create day IDs for the cleaned calendar
# ------------------------------------------------------------

calendar_clean = calendar_clean.copy()

# Sort calendar by date
calendar_clean = calendar_clean.sort_values("date").reset_index(drop=True)

# Create M5 day IDs: d_1, d_2, ..., d_1913
calendar_clean["d"] = [
    f"d_{i}"
    for i in range(1, len(calendar_clean) + 1)
]

# ------------------------------------------------------------
# 2. Check the calendar
# ------------------------------------------------------------

print("\nCalendar columns:")
print(calendar_clean.columns.tolist())

print("\nCalendar shape:")
print(calendar_clean.shape)

print("\nFirst 5 calendar rows:")
display(calendar_clean.head())

# ------------------------------------------------------------
# 3. Merge daily sales with calendar
# ------------------------------------------------------------

daily_data = daily_sales_df.merge(
    calendar_clean[
        [
            "d",
            "date",
            "wm_yr_wk",
            "weekday",
            "wday",
            "month",
            "year",
            "event_name_1",
            "event_type_1",
            "event_name_2",
            "event_type_2"
        ]
    ],
    on="d",
    how="left"
)

# ------------------------------------------------------------
# 4. Sort by date
# ------------------------------------------------------------

daily_data["date"] = pd.to_datetime(daily_data["date"])

daily_data = daily_data.sort_values("date").reset_index(drop=True)

# ------------------------------------------------------------
# 5. Display results
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("MERGED DATASET CREATED")
print("=" * 60)

print("\nRows:", daily_data.shape[0])
print("Columns:", daily_data.shape[1])

print("\nColumns:")
print(daily_data.columns.tolist())

print("\nFirst 10 rows:")
display(daily_data.head(10))

print("\nMissing values:")
print(daily_data.isnull().sum())

print("\n" + "=" * 60)
print("DAILY SALES + CALENDAR MERGE COMPLETED")
print("=" * 60)

MERGING DAILY SALES WITH CALENDAR

Calendar columns:
['date', 'wm_yr_wk', 'weekday', 'wday', 'month', 'year', 'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2', 'snap_CA', 'snap_TX', 'snap_WI', 'd']

Calendar shape:
(1969, 14)

First 5 calendar rows:


,date,wm_yr_wk,weekday,wday,month,year,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI,d
0,2011-01-29,11101,Saturday,1,1,2011,No Event,No Event,No Event,No Event,0,0,0,d_1
1,2011-01-30,11101,Sunday,2,1,2011,No Event,No Event,No Event,No Event,0,0,0,d_2
2,2011-01-31,11101,Monday,3,1,2011,No Event,No Event,No Event,No Event,0,0,0,d_3
3,2011-02-01,11101,Tuesday,4,2,2011,No Event,No Event,No Event,No Event,1,1,0,d_4
4,2011-02-02,11101,Wednesday,5,2,2011,No Event,No Event,No Event,No Event,1,0,1,d_5



MERGED DATASET CREATED

Rows: 1913
Columns: 12

Columns:
['d', 'daily_sales', 'date', 'wm_yr_wk', 'weekday', 'wday', 'month', 'year', 'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2']

First 10 rows:


,d,daily_sales,date,wm_yr_wk,weekday,wday,month,year,event_name_1,event_type_1,event_name_2,event_type_2
0,d_1,32631,2011-01-29,11101,Saturday,1,1,2011,No Event,No Event,No Event,No Event
1,d_2,31749,2011-01-30,11101,Sunday,2,1,2011,No Event,No Event,No Event,No Event
2,d_3,23783,2011-01-31,11101,Monday,3,1,2011,No Event,No Event,No Event,No Event
3,d_4,25412,2011-02-01,11101,Tuesday,4,2,2011,No Event,No Event,No Event,No Event
4,d_5,19146,2011-02-02,11101,Wednesday,5,2,2011,No Event,No Event,No Event,No Event
5,d_6,29211,2011-02-03,11101,Thursday,6,2,2011,No Event,No Event,No Event,No Event
6,d_7,28010,2011-02-04,11101,Friday,7,2,2011,No Event,No Event,No Event,No Event
7,d_8,37932,2011-02-05,11102,Saturday,1,2,2011,No Event,No Event,No Event,No Event
8,d_9,32736,2011-02-06,11102,Sunday,2,2,2011,SuperBowl,Sporting,No Event,No Event
9,d_10,25572,2011-02-07,11102,Monday,3,2,2011,No Event,No Event,No Event,No Event



Missing values:
d               0
daily_sales     0
date            0
wm_yr_wk        0
weekday         0
wday            0
month           0
year            0
event_name_1    0
event_type_1    0
event_name_2    0
event_type_2    0
dtype: int64

DAILY SALES + CALENDAR MERGE COMPLETED


In [11]:
# ============================================================
# STEP 8 — CREATE FORECASTING FEATURES
# ============================================================

print("=" * 60)
print("CREATING FORECASTING FEATURES")
print("=" * 60)

# Make sure date is datetime
daily_data["date"] = pd.to_datetime(daily_data["date"])

# ------------------------------------------------------------
# 1. Calendar-based features
# ------------------------------------------------------------

daily_data["year"] = daily_data["date"].dt.year
daily_data["month"] = daily_data["date"].dt.month
daily_data["week"] = daily_data["date"].dt.isocalendar().week.astype(int)
daily_data["day_of_week"] = daily_data["date"].dt.dayofweek
daily_data["day_of_month"] = daily_data["date"].dt.day

# Weekend indicator
daily_data["is_weekend"] = (
    daily_data["day_of_week"] >= 5
).astype(int)

# ------------------------------------------------------------
# 2. Display information
# ------------------------------------------------------------

print("\nForecasting features created:")

print([
    "year",
    "month",
    "week",
    "day_of_week",
    "day_of_month",
    "is_weekend"
])

print("\nDataset shape:")
print(daily_data.shape)

print("\nColumns:")
print(daily_data.columns.tolist())

# ------------------------------------------------------------
# 3. Show sample
# ------------------------------------------------------------

print("\nFirst 10 rows:")
display(
    daily_data[
        [
            "d",
            "date",
            "daily_sales",
            "year",
            "month",
            "week",
            "day_of_week",
            "day_of_month",
            "is_weekend"
        ]
    ].head(10)
)

print("\n" + "=" * 60)
print("FORECASTING FEATURES CREATED SUCCESSFULLY")
print("=" * 60)

CREATING FORECASTING FEATURES

Forecasting features created:
['year', 'month', 'week', 'day_of_week', 'day_of_month', 'is_weekend']

Dataset shape:
(1913, 16)

Columns:
['d', 'daily_sales', 'date', 'wm_yr_wk', 'weekday', 'wday', 'month', 'year', 'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2', 'week', 'day_of_week', 'day_of_month', 'is_weekend']

First 10 rows:


,d,date,daily_sales,year,month,week,day_of_week,day_of_month,is_weekend
0,d_1,2011-01-29,32631,2011,1,4,5,29,1
1,d_2,2011-01-30,31749,2011,1,4,6,30,1
2,d_3,2011-01-31,23783,2011,1,5,0,31,0
3,d_4,2011-02-01,25412,2011,2,5,1,1,0
4,d_5,2011-02-02,19146,2011,2,5,2,2,0
5,d_6,2011-02-03,29211,2011,2,5,3,3,0
6,d_7,2011-02-04,28010,2011,2,5,4,4,0
7,d_8,2011-02-05,37932,2011,2,5,5,5,1
8,d_9,2011-02-06,32736,2011,2,5,6,6,1
9,d_10,2011-02-07,25572,2011,2,6,0,7,0



FORECASTING FEATURES CREATED SUCCESSFULLY
